# Image Composite Inconsistency Analyzer

**Digital Image Processing Research Prototype**  
**Dataset:** iHarmony4 (HCOCO / HAdobe5k / HFlickr / Hday2night)

---

## Pipeline Overview

| Stage | Description |
|-------|-------------|
| **1. Preprocessing** | Load & aspect-ratio-preserving resize (longest side = 512 px) |
| **2. Region Extraction** | Foreground, background, and boundary band $E$ via morphological ops |
| **3. Color Analysis** | Mean RGB / HSV / CIELAB per region, $\Delta(\text{fg}, \text{bg})$ gap |
| **4. Edge Analysis** | Sobel gradient magnitude & Canny edges within boundary band $E$ |
| **5. Visual Output** | Publication-ready subplot grid + formatted metrics table |

**Inputs:** `real.jpg` ($I_r$), `composite.jpg` ($I_c$), `mask.png` ($M$)  
**Outputs:** `prototype_result.png`, console metrics table

## Imports & Configuration

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"]  = 10

# -- Configuration ---------------------------------------------------------
MAX_PX       = 512          # resize longest side to this
MORPH_KSIZE  = 5            # morphological kernel side length
CANNY_LOW    = 100          # Canny lower threshold
CANNY_HIGH   = 200          # Canny upper threshold
SOBEL_KSIZE  = 3            # Sobel kernel size
OUTPUT_FILE  = "prototype_result.png"

print("OpenCV version:", cv2.__version__)
print("NumPy  version:", np.__version__)

## 0. Synthetic Test-Image Generator

If `real.jpg`, `composite.jpg`, and `mask.png` are not found in the working directory, this cell generates a **deterministic synthetic test set** so the full pipeline can be verified without downloading iHarmony4.

- `real.jpg` -- gradient background + warm elliptical foreground  
- `composite.jpg` -- same background, but the foreground has a deliberately shifted hue/brightness  
- `mask.png` -- binary mask isolating the elliptical foreground

In [ ]:
def generate_synthetic_test_images(directory="."):
    """Create deterministic synthetic test images for pipeline verification."""
    H, W = 480, 640
    rng = np.random.RandomState(42)

    # Background: horizontal colour gradient
    bg = np.zeros((H, W, 3), dtype=np.uint8)
    for x in range(W):
        t = x / W
        bg[:, x] = [int(180 * (1 - t)), int(130 + 50 * t), int(80 + 120 * t)]

    # Foreground ellipse mask
    mask = np.zeros((H, W), dtype=np.uint8)
    cv2.ellipse(mask, (W // 2, H // 2), (120, 80), 0, 0, 360, 255, -1)

    # Real image: warm-coloured ellipse on the gradient
    real = bg.copy()
    real[mask > 0] = np.array([60, 180, 220], dtype=np.uint8)   # warm orange (BGR)
    noise = rng.randint(-8, 9, real.shape, dtype=np.int16)
    real = np.clip(real.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # Composite image: intentionally mismatched foreground
    comp = bg.copy()
    comp[mask > 0] = np.array([180, 100, 90], dtype=np.uint8)   # cool blue (BGR)
    noise2 = rng.randint(-8, 9, comp.shape, dtype=np.int16)
    comp = np.clip(comp.astype(np.int16) + noise2, 0, 255).astype(np.uint8)

    # Write to disk
    cv2.imwrite(os.path.join(directory, "real.jpg"),      real)
    cv2.imwrite(os.path.join(directory, "composite.jpg"), comp)
    cv2.imwrite(os.path.join(directory, "mask.png"),       mask)
    print("[INFO] Synthetic test images generated -> real.jpg, composite.jpg, mask.png")


# Auto-generate if files are missing
needed = ["real.jpg", "composite.jpg", "mask.png"]
if not all(os.path.isfile(f) for f in needed):
    print("[WARN] Input images not found -- generating synthetic test set ...")
    generate_synthetic_test_images(".")
else:
    print("[OK] All input images found.")

## 1. Preprocessing

Load all three images and resize them so their **longest side is 512 pixels** while maintaining the aspect ratio. The mask is re-binarised after resizing to eliminate interpolation artefacts.

In [ ]:
def resize_longest_side(image, max_px=512):
    """Resize so the longest side equals max_px; preserve aspect ratio."""
    h, w = image.shape[:2]
    scale = max_px / max(h, w)
    if scale >= 1.0:
        return image
    new_w, new_h = int(w * scale), int(h * scale)
    return cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)


# Load
I_r = cv2.imread("real.jpg")
I_c = cv2.imread("composite.jpg")
M   = cv2.imread("mask.png", cv2.IMREAD_GRAYSCALE)

assert I_r is not None, "Cannot read real.jpg"
assert I_c is not None, "Cannot read composite.jpg"
assert M   is not None, "Cannot read mask.png"

# Resize
I_r = resize_longest_side(I_r, MAX_PX)
I_c = resize_longest_side(I_c, MAX_PX)
M   = resize_longest_side(M,   MAX_PX)

# Re-binarise mask after potential interpolation
_, M = cv2.threshold(M, 127, 255, cv2.THRESH_BINARY)

# Force identical spatial dimensions (mask may differ by +/- 1 px)
target_h, target_w = I_r.shape[:2]
if I_c.shape[:2] != (target_h, target_w):
    I_c = cv2.resize(I_c, (target_w, target_h), interpolation=cv2.INTER_AREA)
if M.shape[:2] != (target_h, target_w):
    M = cv2.resize(M, (target_w, target_h), interpolation=cv2.INTER_NEAREST)
    _, M = cv2.threshold(M, 127, 255, cv2.THRESH_BINARY)

print("[PREPROCESS] Loaded & resized -> %d x %d px" % (target_w, target_h))
print("  I_r shape:", I_r.shape, " dtype:", I_r.dtype)
print("  I_c shape:", I_c.shape, " dtype:", I_c.dtype)
print("  M   shape:", M.shape,   " dtype:", M.dtype, " unique:", np.unique(M))

## 2. Region Extraction

From the binary mask $M$, we derive three analysis regions:

| Region | Definition |
|--------|------------|
| **Foreground** | $M$ (white pixels) |
| **Background** | $\overline{M}$ (inverted mask) |
| **Boundary Band** $E$ | $\text{dilate}(M, K) - \text{erode}(M, K)$ with $K$ = 5x5 square kernel |

In [ ]:
# Build morphological kernel
kernel = np.ones((MORPH_KSIZE, MORPH_KSIZE), np.uint8)

# Foreground & background masks
fg_mask = M
bg_mask = cv2.bitwise_not(M)

# Boundary band: E = dilate(M) - erode(M)
dilated = cv2.dilate(M, kernel, iterations=1)
eroded  = cv2.erode(M,  kernel, iterations=1)
E       = cv2.subtract(dilated, eroded)

fg_count = np.count_nonzero(fg_mask)
bg_count = fg_mask.size - fg_count
bd_count = np.count_nonzero(E)

print("[REGIONS]")
print("  Foreground:     %6d px" % fg_count)
print("  Background:     %6d px" % bg_count)
print("  Boundary band:  %6d px  (kernel %dx%d)" % (bd_count, MORPH_KSIZE, MORPH_KSIZE))

# Quick visual check
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(fg_mask, cmap="gray"); axes[0].set_title("Foreground Mask $M$")
axes[1].imshow(bg_mask, cmap="gray"); axes[1].set_title("Background Mask $\\overline{M}$")
axes[2].imshow(E,       cmap="gray"); axes[2].set_title("Boundary Band $E$")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Color & Illumination Analysis

For **both** the real image $I_r$ and the composite image $I_c$, we compute the **mean colour** of the foreground and background regions in three colour spaces:

- **RGB** (perceptually intuitive)
- **HSV** (separates hue, saturation, value/brightness)
- **CIELAB** (perceptually uniform -- $\Delta E$ is meaningful)

The **gap** $\Delta = |\mu_{\text{fg}} - \mu_{\text{bg}}|$ quantifies how much the foreground "stands out" from the background. In a well-harmonised image (real), this gap should be smaller than in a poorly-composited image.

In [ ]:
def mean_in_mask(image, mask):
    """Per-channel mean of image where mask > 0."""
    return np.array([
        cv2.mean(image[:, :, c], mask=mask)[0]
        for c in range(image.shape[2])
    ])


def colour_stats(image_bgr, fg_mask, bg_mask):
    """
    Mean colour (RGB / HSV / CIELAB) for FG and BG regions.
    Returns dict: fg_rgb, bg_rgb, gap_rgb, fg_hsv, ... (9 keys)
    """
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2Lab)

    stats = {}
    for name, img in [("rgb", rgb), ("hsv", hsv), ("lab", lab)]:
        fg_mean = mean_in_mask(img, fg_mask)
        bg_mean = mean_in_mask(img, bg_mask)
        stats["fg_"  + name] = fg_mean
        stats["bg_"  + name] = bg_mean
        stats["gap_" + name] = np.abs(fg_mean - bg_mean)
    return stats


# Compute for both images
cs_real = colour_stats(I_r, fg_mask, bg_mask)
cs_comp = colour_stats(I_c, fg_mask, bg_mask)

print("[COLOR] Analysis complete for RGB / HSV / CIELAB")

## 4. Edge Analysis

We examine edge behaviour at the **boundary band** $E$ -- the transition zone between foreground and background. A poorly composited image tends to show:

- **Higher** Sobel gradient magnitudes (sharp colour/luminance jumps)
- **Higher** Canny edge density (more pronounced discontinuities)

Both the **Sobel operator** (gradient magnitude $|\nabla I| = \sqrt{G_x^2 + G_y^2}$) and the **Canny edge detector** (thresholds 100/200) are applied to grayscale versions of both images.

In [ ]:
def edge_analysis(image_bgr, boundary_mask):
    """
    Sobel gradient magnitude & Canny edge analysis.
    Returns: sobel_mag, canny_map, density_dict
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Sobel gradient magnitude
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=SOBEL_KSIZE)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=SOBEL_KSIZE)
    sobel_mag = cv2.magnitude(gx, gy)

    # Normalise to [0,255] for density metric
    sobel_norm = cv2.normalize(sobel_mag, None, 0, 255,
                               cv2.NORM_MINMAX).astype(np.uint8)

    # Canny edge detector
    canny_map = cv2.Canny(gray, CANNY_LOW, CANNY_HIGH)

    # Mean density only inside boundary band E
    bd = boundary_mask > 0
    if np.any(bd):
        density_sobel = float(np.mean(sobel_norm[bd]))
        density_canny = float(np.mean(canny_map[bd] / 255.0))
    else:
        density_sobel = 0.0
        density_canny = 0.0

    return sobel_mag, canny_map, {"sobel": density_sobel, "canny": density_canny}


# Compute for both images
sobel_r, canny_r, ed_real = edge_analysis(I_r, E)
sobel_c, canny_c, ed_comp = edge_analysis(I_c, E)

print("[EDGE] Sobel density in E -- Real: %.2f  |  Comp: %.2f" %
      (ed_real["sobel"], ed_comp["sobel"]))
print("[EDGE] Canny density in E -- Real: %.4f  |  Comp: %.4f" %
      (ed_real["canny"], ed_comp["canny"]))

## 5. Visual Output (Publication Figure)

A **3 x 3 subplot grid** suitable for direct inclusion in a research paper:

| | Col 0 | Col 1 | Col 2 |
|---|---|---|---|
| **Row 0** | Composite $I_c$ | Real $I_r$ | Mask $M$ |
| **Row 1** | Sobel (Composite) | Sobel (Real) | Boundary Band $E$ |
| **Row 2** | Canny (Composite) | Canny (Real) | Pipeline Notes |

In [ ]:
def bgr2rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


fig = plt.figure(figsize=(16, 15))
fig.suptitle("Image Composite Inconsistency Analysis",
             fontsize=18, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(3, 3, hspace=0.25, wspace=0.15)

panels = [
    # (gridspec slot,  data,     title,                        cmap)
    (gs[0, 0], bgr2rgb(I_c),    "Composite Image $I_c$",      None),
    (gs[0, 1], bgr2rgb(I_r),    "Real Image $I_r$",           None),
    (gs[0, 2], M,               "Foreground Mask $M$",         "gray"),
    (gs[1, 0], sobel_c,         "Sobel Gradient -- Composite", "inferno"),
    (gs[1, 1], sobel_r,         "Sobel Gradient -- Real",      "inferno"),
    (gs[1, 2], E,               "Boundary Band $E$",           "gray"),
    (gs[2, 0], canny_c,         "Canny Edges -- Composite",    "gray"),
    (gs[2, 1], canny_r,         "Canny Edges -- Real",         "gray"),
]

for spec, data, title, cmap in panels:
    ax = fig.add_subplot(spec)
    ax.imshow(data, cmap=cmap, aspect="equal")
    ax.set_title(title, fontsize=11, pad=6)
    ax.axis("off")

# Bottom-right cell: methodology note
ax_text = fig.add_subplot(gs[2, 2])
ax_text.axis("off")
note = (
    "Pipeline notes\n"
    "---------------------\n"
    "  Resize: longest -> 512 px\n"
    "  Morphology: 5x5 kernel\n"
    "  Sobel: 3x3, mag(Gx,Gy)\n"
    "  Canny: thresh 100 / 200\n"
    "  Boundary band E =\n"
    "    dilate(M) - erode(M)"
)
ax_text.text(0.05, 0.95, note, transform=ax_text.transAxes,
             fontsize=10, verticalalignment="top",
             fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f0f0",
                       edgecolor="#cccccc"))

plt.savefig(OUTPUT_FILE, dpi=180, bbox_inches="tight", pad_inches=0.3)
plt.show()

print("\n[OUTPUT] Figure saved ->", OUTPUT_FILE)

## 6. Summary Metrics Table

Formatted comparison of all computed metrics: **Real** ($I_r$) vs **Composite** ($I_c$).

In [ ]:
def fmt(arr):
    """Format a 3-element array as (x.x, y.y, z.z)."""
    return "(%5.1f, %5.1f, %5.1f)" % (arr[0], arr[1], arr[2])

sep  = "=" * 88
thin = "-" * 88

print(sep)
print("  COMPOSITE INCONSISTENCY METRICS  --  Real (I_r)  vs  Composite (I_c)")
print(sep)

# -- Color & Illumination --------------------------------------------------
print("\n  [COLOR & ILLUMINATION -- mean per region]\n")
print("  %-6s %-5s %22s   %22s   %16s  %16s" %
      ("Space", "Reg", "Real", "Composite", "Gap(Real)", "Gap(Comp)"))
print("  " + thin)

for space in ("rgb", "hsv", "lab"):
    label = space.upper()
    fg_r  = fmt(cs_real["fg_"  + space])
    bg_r  = fmt(cs_real["bg_"  + space])
    fg_c  = fmt(cs_comp["fg_"  + space])
    bg_c  = fmt(cs_comp["bg_"  + space])
    gap_r = fmt(cs_real["gap_" + space])
    gap_c = fmt(cs_comp["gap_" + space])

    print("  %-6s %-5s %22s   %22s   %16s  %16s" %
          (label, "FG", fg_r, fg_c, gap_r, gap_c))
    print("  %-6s %-5s %22s   %22s" % ("", "BG", bg_r, bg_c))

# -- Edge Density ----------------------------------------------------------
print("\n  [EDGE DENSITY -- boundary band E]\n")
print("  %-20s %10s   %10s   %10s" % ("Detector", "Real", "Composite", "Delta"))
print("  " + thin)

for key, label in [("sobel", "Sobel magnitude"), ("canny", "Canny binary")]:
    vr    = ed_real[key]
    vc    = ed_comp[key]
    delta = abs(vc - vr)
    print("  %-20s %10.3f   %10.3f   %10.3f" % (label, vr, vc, delta))

print("\n" + sep)

## Interpretation Guide

| Metric | What it reveals | Expected for bad composite |
|--------|----------------|---------------------------|
| **RGB/HSV/LAB Gap** | Colour mismatch between FG & BG | Larger gap than real image |
| **LAB Gap (L channel)** | Luminance inconsistency | Foreground too bright/dark vs background |
| **HSV Gap (H channel)** | Hue shift | Foreground has different colour temperature |
| **Sobel density (E)** | Gradient strength at boundary | Higher = sharper, unnatural transition |
| **Canny density (E)** | Edge prevalence at boundary | Higher = more visible seam |

---

*To use with your own iHarmony4 data, simply place `real.jpg`, `composite.jpg`, and `mask.png` in the same directory as this notebook and re-run all cells.*